In [58]:
import sys
import random
import math
import logging
import json
import os

logging.basicConfig(
    filename = "game.log",
    level = logging.INFO,
    format =  "%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

class Game:

    def __init__(self):
        self.last_difficulty = None
        self.state = "menu"
        self.game_properties = {
            "menu": self.menu,
            "difficulty": self.difficulty,
            "stats": self.view_stats,
            "easy": lambda: self.game_play(1, 10, math.inf),
            "medium": lambda: self.game_play(1, 100, 5),
            "hard": lambda: self.game_play(1, 1000, 3),
            "continue": self.game_end
        }
        self.stats = {}
        
    def create_stats(self):
        if not os.path.exists("stats.json"):
            with open("stats.json", "w") as file:
                json.dump({
                    "games_played": 0,
                    "wins": 0,
                    "wins_with_attempts": 0,
                    "losses": 0,
                    "games_easy": 0,
                    "easy_wins": 0,
                    "games_medium": 0,
                    "medium_wins": 0,
                    "medium_wins_with_attempts": 0,
                    "medium_losses": 0,
                    "games_hard": 0,
                    "hard_wins": 0,
                    "hard_wins_with_attempts": 0,
                    "hard_losses": 0,
                }, file)

    def load_stats(self):
        with open("stats.json", "r") as file:
            self.stats = json.load(file)

    def save_stats(self):
        with open("stats.json", "w") as file:
            json.dump(self.stats, file)

    def run(self):
        self.create_stats()
        self.load_stats()
        logger.info("Game started")
        while self.state:
            self.state = self.game_properties[self.state]()
            
    def menu(self):
        print("-" * 50)
        print("[Play]\n[View Stats]\n[Exit]")
        print("-" * 50)
        while True:
            menu_input = input().lower()
            if menu_input in {"play"}:
                return "difficulty"
            elif menu_input in {"view stats"}:
                return "stats"
            elif menu_input in {"exit"}:
                print("-" * 50)
                print("Thx for playing!")
                print("-" * 50)
                logger.info("Player exited the game")
                logging.shutdown()
                sys.exit()
            else:
                print("-" * 50)
                print("Invalid command!")
                print("-" * 50)

    def view_stats(self):
        win_rate = round((self.stats["wins"] / self.stats["games_played"]) * 100, 2)
        all_stats = [f"Games: {self.stats['games_played']}",
                    "-" * 50,
                    f"Wins: {self.stats['wins']}",
                    "-" * 50,
                    f"Wins with attempts left: {self.stats['wins_with_attempts']}",
                    "-" * 50,
                    f"Losses: {self.stats['losses']}",
                    "-" * 50,
                    f"Win rate: {win_rate}%",
                    "-" * 50,
                    f"Games played on difficulty: ",
                    "-" * 50,
                    f"Easy: {self.stats['games_easy']}",
                    "-" * 50,
                    f"Medium: {self.stats['games_medium']}",
                    "-" * 50,
                    f"Hard: {self.stats['games_hard']}",
                    "-" * 50,
                    f"Wins on difficulty: ",
                    "-" * 50,
                    f"Easy: {self.stats['easy_wins']}",
                    "-" * 50,
                    f"Medium: {self.stats['medium_wins']}",
                    "-" * 50,
                    f"Hard: {self.stats['hard_wins']}",
                    "-" * 50,
                    f"Wins with attempts on difficulty: ",
                    "-" * 50,
                    f"Medium: {self.stats['medium_wins_with_attempts']}",
                    "-" * 50,
                    f"Hard: {self.stats['hard_wins_with_attempts']}",
                    "-" * 50,
                    f"Losses on difficulty: ",
                    "-" * 50,
                    f"Medium: {self.stats['medium_losses']}",
                    "-" * 50,
                    f"Hard: {self.stats['hard_losses']}",
                    "-" * 50]
        print("-" * 50)
        print("\n".join(all_stats))
        print("[Back]")
        print("-" * 50)
        while True:
            back_input = input().lower()
            if back_input in {"back"}:
                return "menu"
            else:
                print("-" * 50)
                print("Invalid command!")
                print("-" * 50)
                
    def difficulty(self):
        print("-" * 50)
        print("[Easy]\n[Medium]\n[Hard]\n[Return]")
        print("-" * 50)
        while True:
            difficulty_input = input().lower()
            if difficulty_input in {"easy"}:
                self.last_difficulty = "easy"
                logger.info("Difficulty selected: easy")
                return "easy"
            elif difficulty_input in {"medium"}:
                self.last_difficulty = "medium"
                logger.info("Difficulty selected: medium")
                return "medium"
            elif difficulty_input in {"hard"}:
                self.last_difficulty = "hard"
                logger.info("Difficulty selected: hard")
                return "hard"
            elif difficulty_input in {"return"}:
                logger.info("Returned to menu")
                return "menu"
            else:
                print("Invalid choice!")

    def game_prompts(self, random_number, guess):
        difference = abs(random_number - guess)
        if (25 >= difference > 15):
            print("Cold!")
        elif (100 >= difference > 25):
            print("Super cold!")
        elif (difference > 100):
            print("Extremely cold!")
        elif (15 >= difference > 10):
            print("Warm!")
        elif (self.last_difficulty != "easy" and 10 >= difference > 5):
            print("Very warm!")
        elif (self.last_difficulty != "easy" and 5 >= difference > 0):
            print("Really hot!")

    def stat_check(self, diff, attempts, result):
        self.stats["games_played"] += 1
        if (diff == "easy" and result == "win"):
            self.stats["games_easy"] += 1
            self.stats["wins"] += 1
            self.stats["easy_wins"] += 1
        elif (diff == "medium" and result == "win"):
            self.stats["games_medium"] += 1
            self.stats["wins"] += 1
            self.stats["medium_wins"] += 1
            if (attempts > 0 and attempts != math.inf):
                self.stats["wins_with_attempts"] += 1
                self.stats["medium_wins_with_attempts"] += 1
        elif (diff == "hard" and result == "win"):
            self.stats["games_hard"] += 1 
            self.stats["wins"] += 1
            self.stats["hard_wins"] += 1
            if (attempts > 0 and attempts != math.inf):
                self.stats["wins_with_attempts"] += 1
                self.stats["hard_wins_with_attempts"] += 1

        if (diff == "medium" and result == "loss"):
            self.stats["games_medium"] += 1
            self.stats["losses"] += 1
            self.stats["medium_losses"] += 1
        elif (diff == "hard" and result == "loss"):
            self.stats["games_hard"] += 1 
            self.stats["losses"] += 1
            self.stats["hard_losses"] += 1
            
    def game_play(self, min_range, max_range, attempts_limit):
        random_number = random.randint(min_range, max_range)
        print("-" * 50)
        print("Guess!")
        while True:
            try:
                guess = int(input())
                    
                if (guess > random_number):
                    print("-" * 50)
                    print ("Too high!")
                    self.game_prompts(random_number, guess)
                    print("-" * 50)
                    attempts_limit -= 1
                elif (guess < random_number):
                    print("-" * 50)
                    print ("Too low!")
                    self.game_prompts(random_number, guess)
                    print("-" * 50)
                    attempts_limit -= 1
                elif (guess == random_number):
                    self.stat_check(self.last_difficulty, attempts_limit, "win")
                    self.save_stats()
                    logger.info(
                        f"Win | difficulty = {self.last_difficulty} | number = {random_number}"
                    )
                    print("-" * 50)
                    print(f"Great guess!\nTry again?\nYes/No")
                    print("-" * 50)
                    return "continue"
                        
                if (attempts_limit != math.inf and attempts_limit > 0):
                    print(f"Attempts left: {attempts_limit}")
                    print("-" * 50)
                    continue
                elif (attempts_limit == 0):
                    self.stat_check(self.last_difficulty, attempts_limit, "loss")
                    self.save_stats()
                    logger.info(
                        f"Loss | difficulty = {self.last_difficulty} | number = {random_number}"
                    )
                    print(f"No attempts left!\nThe number was {random_number}\nTry again?\nYes/No")
                    print("-" * 50)
                    return "continue"
                        
            except ValueError:
                print("-" * 50)
                print("Invalid guess!")
                print("-" * 50)
                    
    def game_end(self):
        while True:
            choice = input().lower()
            if (choice == "yes"):
                logger.info("Player started a new game on the same difficulty")
                return self.last_difficulty
            elif (choice == "no"):
                logger.info("Player ended the game")
                return "menu"
            else:
                print("-" * 50)
                print("Invalid command!")
                print("Try again?\nYes/No")
                print("-" * 50)
                    
game = Game()
game.run()

--------------------------------------------------
[Play]
[View Stats]
[Exit]
--------------------------------------------------


 view stats


--------------------------------------------------
Games: 13
--------------------------------------------------
Wins: 6
--------------------------------------------------
Wins with attempts left: 2
--------------------------------------------------
Losses: 7
--------------------------------------------------
Win rate: 46.15%
--------------------------------------------------
Games played on difficulty: 
--------------------------------------------------
Easy: 4
--------------------------------------------------
Medium: 5
--------------------------------------------------
Hard: 4
--------------------------------------------------
Wins on difficulty: 
--------------------------------------------------
Easy: 4
--------------------------------------------------
Medium: 1
--------------------------------------------------
Hard: 1
--------------------------------------------------
Wins with attempts on difficulty: 
--------------------------------------------------
Medium: 1
----------------

 back


--------------------------------------------------
[Play]
[View Stats]
[Exit]
--------------------------------------------------


 exit


--------------------------------------------------
Thx for playing!
--------------------------------------------------


SystemExit: 

In [54]:
'''
added logging
added stats tracking
added a view stats tab
polished the buttons and tabs to be more readible and look like buttons are seperate and menus are different from each other
changed logic so that games are counted only the game_play method instead of being split between the diff method and the game_end method
changed stats from being saved only on exiting the program, now they are saved immediately after the game is won or lost, so it updates in real time
added a method to do the checks for which stats has to be updated instead of spamming check in the game_play method
added more stats in the view stats tab
polished the look in the view stats tab
'''
#add matching mode

'\nadded logging\nadded stats tracking\nadded a view stats tab\npolished the buttons and tabs to be more readible and look like buttons are seperate and menus are different from each other\nchanged logic so that games are counted only the game_play method instead of being split between the diff method and the game_end method\nchanged stats from being saved only on exiting the program, now they are saved immediately after the game is won or lost, so it updates in real time\nadded a method to do the checks for which stats has to be updated instead of spamming check in the game_play method\nadded more stats in the view stats tab\npolished the look in the view stats tab\n'

In [ ]:
import unittest
from unittest.mock import patch

class TestGame(unittest.TestCase):

    @patch("builtins.input", side_effect = ["asd", "213","play"])
    def test_menu_play(self, mock_input):
        game = Game()
        result = game.menu()
        self.assertEqual(result, "difficulty")

    def test_menu_exit(self):
        game = Game()
        with patch("builtins.input", side_effect = ["sdsa", "321", "exit"]):
            with self.assertRaises(SystemExit):
                game.menu()

    @patch("builtins.input", side_effect = ["asd", "123", "return"])
    def test_return_invalid_then_correct(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "menu")
    
    @patch("builtins.input", return_value = "easy")
    def test_difficulty_easy(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "easy")
        self.assertEqual(game.last_difficulty, "easy")

    @patch("builtins.input", side_effect = ["wrong", "medium"])
    def test_invalid_then_medium(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "medium")
        self.assertEqual(game.last_difficulty, "medium")

    @patch("random.randint", return_value = 7)
    @patch("builtins.input", side_effect = ["2", "3", "9", "7"])
    def test_game_play_win_on_correct_guess(self, mock_input, mock_randint):
        game = Game()
        result = game.game_play(1, 10, math.inf)
        self.assertEqual(result, "continue")

    @patch("builtins.input", side_effect = ["asdas", "wrong", "hard"])
    def test_invalid_difficulty_then_hard(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "hard")
        self.assertEqual(game.last_difficulty, "hard")

    @patch("random.randint", return_value = 44)
    @patch("builtins.input", side_effect = ["2", "45", "60", "30", "66"])
    def test_exceeding_attempt_limits(self, mock_input, mock_randint):
        game = Game()
        result = game.game_play(1, 100, 5)
        self.assertEqual(result, "continue")

    @patch("builtins.input", side_effect = ["sa", "123", "no"])
    def test_invalid_game_end_input_return(self, mock_input):
        game = Game()
        result = game.game_end()
        self.assertEqual(result, "menu")

    @patch("builtins.input", side_effect = ["sda", "213", "yes"])
    def test_invalid_game_end_input_continue_last_difficulty(self, mock_input):
        game = Game()
        game.last_difficulty = "hard"
        result = game.game_end()
        self.assertEqual(result, "hard")

In [ ]:
#@patch - switches given part in the code
#"builtins.input" - switches input in the code
#"random.randint" - switches the random_number in the code
#side_effect - used for multiple inputs
#return_value = "easy" - the input that "builtins.input" switches, used for a value that is given once
#mock_input - is just needed to be there
#result = takes the real value from the code
#self.assertEqual(result, "easy") - compares result to the given string - "easy" if it doesnt match an error is given
#self.assertEqual(game.last_difficulty, "easy") - checks the the code's memory variable

In [ ]:
suite = unittest.TestLoader().loadTestsFromTestCase(TestGame)
unittest.TextTestRunner(verbosity = 2).run(suite)